# 04 · Étude de faisabilité · projection, regroupement, mesure

**Ce que fait ce notebook.** Il répond à la question centrale de la mission : les produits d'une même
catégorie se rapprochent-ils spontanément, une fois traduits en nombres ? On projette en deux
dimensions pour regarder, puis on mesure l'accord entre des groupes formés sans étiquettes et les
vraies catégories.

**Ce qu'il établit.** L'information est bien présente. Les caractéristiques visuelles issues d'un
réseau pré-entraîné structurent le catalogue nettement mieux que le texte, et bien mieux que SIFT,
qui reste proche du hasard sur les mêmes photographies.

In [1]:
import sys

sys.path.insert(0, "..")
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

## Protocole

L'étude est non supervisée : les catégories ne servent qu'à colorier les graphiques et à mesurer
l'accord final. Elles n'entrent jamais dans la construction des représentations, ce qui autorise à
travailler sur les 1 050 produits sans risque de fuite.

Une analyse en composantes principales ramène chaque représentation à 50 dimensions, puis t-SNE la
met en plan. Chaque produit est d'abord ramené à une longueur unitaire, faute de quoi une
description longue occuperait mécaniquement une position plus éloignée de l'origine.

In [2]:
from src.faisabilite import etudier
from src.pipeline import LABEL_COL, TEXT_COL, load
from src.representations import IMAGE, TEXTE, obtenir

df = load()
categories = df[LABEL_COL].tolist()
entrees = {"texte": df[TEXT_COL].tolist(), "image": df["uniq_id"].tolist()}

etudes = {}
for famille, registre in (("texte", TEXTE), ("image", IMAGE)):
    for nom in registre:
        X, _ = obtenir(nom, entrees[famille])
        etudes[nom] = etudier(X, categories)
        print(f"  {nom:22s} ARI projection {etudes[nom]['ARI projection']:.3f}")

  Comptage de mots       ARI projection 0.306


  Comptage + bigrammes   ARI projection 0.316


  TF-IDF                 ARI projection 0.325


  Word2Vec               ARI projection 0.300


  BERT                   ARI projection 0.316


  USE                    ARI projection 0.440


  SIFT                   ARI projection 0.044


  CNN (VGG16)            ARI projection 0.509


## Indice de Rand ajusté · lecture et contresens

L'indice compare deux découpages d'un même ensemble. Il vaut 1 lorsqu'ils coïncident, 0 lorsque leur
accord n'excède pas le hasard. L'ajustement compte : avec sept groupes de tailles voisines, deux
découpages tirés au sort présentent déjà un accord apparent que l'indice brut compterait à tort.

**Un indice de 0,51 ne signifie pas que 51 % des produits sont bien classés.** Ce n'est pas une
proportion, et les groupes formés n'ont d'ailleurs pas de nom : rien ne dit lequel correspond aux
montres.

Nous rapportons la mesure deux fois : sur la projection, que nous avons regardée, et sur la
représentation complète, parce que t-SNE déforme et qu'il serait commode de ne retenir que le plus
flatteur des deux chiffres.

In [3]:
tableau = pd.DataFrame(
    [
        {
            "Représentation": nom,
            "Dimensions": e["dimensions"],
            "ARI (projection 2D)": e["ARI projection"],
            "ARI (espace complet)": e["ARI représentation complète"],
        }
        for nom, e in etudes.items()
    ]
).sort_values("ARI (projection 2D)", ascending=False)

tableau

,Représentation,Dimensions,ARI (projection 2D),ARI (espace complet)
7,CNN (VGG16),512,0.5095,0.5398
5,USE,512,0.4399,0.3328
2,TF-IDF,5000,0.3253,0.2139
1,Comptage + bigrammes,5000,0.3161,0.2272
4,BERT,768,0.3161,0.2878
0,Comptage de mots,2444,0.3061,0.2695
3,Word2Vec,300,0.3001,0.2073
6,SIFT,256,0.0445,0.0555


## Lecture du tableau

**Sur les mêmes photographies**, VGG16 atteint 0,51 quand SIFT reste proche du hasard. SIFT décrit
des motifs locaux, un angle ou une texture, utiles pour reconnaître qu'une même scène a été
photographiée deux fois, mais qui ne disent rien de ce qu'est l'objet.

**VGG16 est la seule représentation dont l'accord est meilleur avant réduction qu'après.** Sa
structure ne doit donc rien à t-SNE. Pour les représentations textuelles, l'écart va dans l'autre
sens : lue seule, la colonne de gauche leur accorderait une netteté que l'espace d'origine ne
confirme pas.

**Le comptage simple fait pratiquement jeu égal avec BERT.** Sur des fiches de spécifications, où le
vocabulaire est très discriminant et la syntaxe presque absente, comprendre le contexte n'apporte
presque rien de plus que compter les mots.

In [4]:
meilleure = tableau.iloc[0]["Représentation"]
etude = etudes[meilleure]

croise = pd.crosstab(
    pd.Series(categories, name="catégorie réelle"),
    pd.Series(etude["groupes_projection"], name="groupe"),
)
print(f"{meilleure} — chaque catégorie a-t-elle son groupe ?")
croise

CNN (VGG16) — chaque catégorie a-t-elle son groupe ?


groupe,0,1,2,3,4,5,6
catégorie réelle,,,,,,,
Baby Care,15,6,16,2,0,3,108
Beauty and Personal Care,7,13,6,1,0,120,3
Computers,6,130,10,1,2,1,0
Home Decor & Festive Needs,31,3,96,8,6,3,3
Home Furnishing,71,0,10,0,0,0,69
Kitchen & Dining,8,16,5,106,0,15,0
Watches,0,20,1,0,129,0,0


## Zones de confusion

*Home Furnishing* se scinde presque en deux : une moitié dans son groupe, l'autre dans celui de
*Baby Care*. Regardons ce que contient réellement cette seconde moitié.

In [5]:
groupe_baby = int(croise.loc["Baby Care"].idxmax())
avec = df.assign(groupe=etude["groupes_projection"])

print("Home Furnishing tombés dans le groupe de Baby Care :")
for nom in avec[(avec[LABEL_COL] == "Home Furnishing") & (avec.groupe == groupe_baby)][
    "product_name"
].head(6):
    print("   -", nom[:70])

print()
print("Baby Care de ce même groupe :")
for nom in avec[(avec[LABEL_COL] == "Baby Care") & (avec.groupe == groupe_baby)][
    "product_name"
].head(6):
    print("   -", nom[:70])

Home Furnishing tombés dans le groupe de Baby Care :
   - Riva Carpets Cotton Free Bath Mat Classic Loop Shag Bathmat_RI-527
   - JMD Printed Cushions Cover
   - Kripa's Printed Cushions Cover
   - Rama Floral Single Quilts & Comforters Pink-Red
   - Rama Floral Single Quilts & Comforters Yellow
   - Artisan Creation Checkered Single Quilts & Comforters Brown

Baby Care de ce même groupe :
   - Sathiyas Cotton Bath Towel
   - Eurospa Cotton Terry Face Towel Set
   - Mom and Kid Baby Girl's Printed Green Top & Pyjama Set
   - Mom and Kid Baby Girl's Printed Blue, Grey Top & Pyjama Set
   - KOHL Wine Bag Yellow
   - CHHOTE JANAB COZY MATTRESS PROTECTOR(SET OF 2)


Des housses de coussin, des couettes et des tapis de bain d'un côté ; des serviettes en coton, des
pyjamas de bébé et des protège-matelas de l'autre. Ce groupe ne correspond à aucune des deux
catégories : il rassemble des **textiles imprimés photographiés à plat**.

L'algorithme a regroupé par matière et par mise en scène, ce qui est ce qu'on lui a demandé de
faire, alors que la nomenclature du site regroupe par usage commercial. Une couette et un pyjama de
bébé n'ont rien en commun pour un acheteur ; ils se ressemblent beaucoup pour un réseau de vision.

**Ce que ce notebook établit.** L'information nécessaire à la catégorisation est présente dans les
données, et suffisamment pour que des groupes cohérents émergent sans qu'aucune étiquette n'ait été
montrée. Les photographies traitées par un réseau pré-entraîné sont la source la plus prometteuse.